In [3]:
#Preprocesamiento del Texto
#Librerías necesarias para ejecutar el script
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import pad_sequences
import pickle
import PyPDF2
import re #Librería para limpieza de texto


#1. Cargar Archivo pdf y Limpiarlo
ruta_archivo = "/content/documento.pdf"
#Se extrae el contenido textual de un archivo pdf y se lo devuelve en bruto.
def extraer_texto_pdf(ruta_archivo):
    texto_completo = ""
    # Abrimos el archivo en modo 'rb' (read binary)
    with open(ruta_archivo, 'rb') as archivo:
        lector_pdf = PyPDF2.PdfReader(archivo)
        # Recorremos cada página del PDF
        for pagina in lector_pdf.pages:
            if pagina.extract_text():
                texto_completo += pagina.extract_text() + " "
    return texto_completo

#Con el texto extraido se realiza un proceso de limpieza y se devuelve el texto procesado
def limpiar_texto(texto_crudo):
    # 1. Pasar todo a minúsculas para unificar (para la red "Gato" y "gato" son distintas si no lo hacemos)
    texto = texto_crudo.lower()
    # 2. Eliminar saltos de línea y caracteres especiales (dejamos solo letras y espacios)
    texto = re.sub(r'[^a-záéíóúñ\s]', ' ', texto)
    # 3. Eliminar espacios en blanco múltiples
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

texto_crudo = extraer_texto_pdf('documento.pdf')
texto = limpiar_texto(texto_crudo)

# 2. Inicializar y entrenar el Tokenizer
# El Tokenizer convertirá cada palabra en un número único.
tokenizer = Tokenizer()
# Ajustamos el tokenizer a nuestro texto (crea el diccionario)
tokenizer.fit_on_texts([texto])

# Obtenemos el tamaño del vocabulario (sumamos 1 por un requisito técnico de Keras para el padding)
vocab_size = len(tokenizer.word_index) + 1
print(f"Tamaño del vocabulario: {vocab_size} palabras únicas.")

# 3. Generar secuencias de N-gramas
# Esto crea las secuencias de aprendizaje.
# Conversión del bloque de texto del archivo pdf procesado a una sola lista inmensa de números
secuencia_completa = tokenizer.texts_to_sequences([texto])[0]

input_sequences = []
#Limite de palabras para predecir la siguiente
LONGITUD_VENTANA = 6
# Recorremos la lista gigante creando "ventanas" de 6 números
for i in range(LONGITUD_VENTANA, len(secuencia_completa)):
    # Extraemos un pedazo de la secuencia
    n_gram_sequence = secuencia_completa[i - LONGITUD_VENTANA : i]
    input_sequences.append(n_gram_sequence)

# 4. Padding (Igualar tamaños)
# La red neuronal necesita que todas las entradas tengan la misma longitud.
# Rellenamos con ceros (0) a la izquierda las secuencias más cortas.
max_sequence_len = max([len(x) for x in input_sequences])
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre'))

# 5. Separar X (entrada) e Y (etiqueta a predecir)
# X son todas las palabras menos la última. Y es la última palabra.
X, y = input_sequences[:,:-1], input_sequences[:,-1]

# Convertimos 'y' al formato One-Hot Encoding (necesario para la capa Softmax)
y = tf.keras.utils.to_categorical(y, num_classes=vocab_size)

# 6. Guardar el Tokenizer para usarlo después en la aplicación final
with open('tokenizer.pkl', 'wb') as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)

print("¡Procesamiento completado!")
print(f"Forma de X (Entradas): {X.shape}")
print(f"Forma de Y (Etiquetas): {y.shape}")

Tamaño del vocabulario: 165 palabras únicas.
¡Procesamiento completado!
Forma de X (Entradas): (255, 5)
Forma de Y (Etiquetas): (255, 165)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [3]:
#LSTM y Entrenamiento
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.optimizers import Adam

# 1. DEFINIR LA ARQUITECTURA
modelo = Sequential()

# Capa 1: Embedding (Incrustación)
# Convierte los números de las palabras en vectores densos para que la red entienda el "significado" o contexto.
modelo.add(Embedding(input_dim=vocab_size, output_dim=10, input_length=max_sequence_len-1))

# Capa 2: LSTM (Memoria a Corto y Largo Plazo)
# El corazón de nuestra red. Las 50 'units' son las celdas de memoria que analizan la secuencia.
modelo.add(LSTM(units=50))

# Capa 3: Capa Densa (Salida)
# Usa la función 'softmax' para darnos una probabilidad.
# Si el vocabulario tiene 20 palabras, nos dará 20 porcentajes. El más alto es la palabra predicha.
modelo.add(Dense(units=vocab_size, activation='softmax'))

# 2. COMPILAR EL MODELO
# Le decimos a la red cómo va a medir sus errores (loss) y cómo va a mejorar (optimizer)
modelo.compile(loss='categorical_crossentropy',
               optimizer=Adam(learning_rate=0.01),
               metrics=['accuracy'])

# Imprimimos un resumen de la arquitectura (¡Guarda esta tabla para tu presentación!)
print("--- RESUMEN DE LA ARQUITECTURA ---")
modelo.summary()

# 3. ENTRENAR EL MODELO
# 'epochs' son las veces que la red repasará el texto para aprender de sus errores.
print("\nIniciando el entrenamiento (esto puede tomar unos segundos)...")
historial = modelo.fit(X, y, epochs=100, verbose=1)
print("¡Entrenamiento finalizado con éxito!")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


--- RESUMEN DE LA ARQUITECTURA ---


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)


Iniciando el entrenamiento (esto puede tomar unos segundos)...
Epoch 1/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 6s 6s/step - accuracy: 0.0000e+00 - loss: 2.7732
Epoch 2/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 0.2632 - loss: 2.7556
Epoch 3/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.2632 - loss: 2.7334
Epoch 4/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 0.2632 - loss: 2.7013
Epoch 5/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 0.2632 - loss: 2.6549
Epoch 6/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.2632 - loss: 2.5916
Epoch 7/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.2632 - loss: 2.5203
Epoch 8/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 0.2632 - loss: 2.4595
Epoch 9/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.2632 - loss: 2.3643
Epoch 10/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.2632 - loss: 2.2383
Epoch 11/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 0.2632 - loss: 2.1131
Epoch 12

In [5]:
#Generador de Autocompletado
import numpy as np

def predecir_siguiente_palabra(modelo, tokenizer, texto_semilla, cantidad_palabras, max_sequence_len):
    """
    Toma una frase inicial y predice las siguientes N palabras usando el modelo LSTM.
    """
    for _ in range(cantidad_palabras):
        # 1. Convertir la frase escrita a números usando nuestro diccionario
        token_list = tokenizer.texts_to_sequences([texto_semilla])[0]

        # 2. Hacer el Padding (rellenar con ceros) para que coincida con lo que espera la red
        # Usamos max_sequence_len - 1 porque la última palabra es la que queremos adivinar
        token_list = tf.keras.utils.pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')

        # 3. Pedirle a la red que prediga (Nos devuelve un vector con probabilidades para cada palabra)
        probabilidades = modelo.predict(token_list, verbose=0)

        # 4. Obtener el índice (número) de la palabra con la probabilidad más alta
        indice_predicho = np.argmax(probabilidades, axis=-1)[0]

        # 5. Buscar qué palabra de nuestro diccionario corresponde a ese número
        palabra_predicha = ""
        for palabra, indice in tokenizer.word_index.items():
            if indice == indice_predicho:
                palabra_predicha = palabra
                break

        # 6. Añadir la palabra predicha a nuestra frase original para la siguiente vuelta
        texto_semilla += " " + palabra_predicha

    return texto_semilla

# --- PRUEBA EN VIVO ---
# Vamos a darle una frase incompleta y pedirle que adivine las siguientes 3 palabras
frase_prueba_1 = "El gato"
resultado_1 = predecir_siguiente_palabra(modelo, tokenizer, frase_prueba_1, cantidad_palabras=3, max_sequence_len=max_sequence_len)
print(f"Usuario escribe: '{frase_prueba_1}'")
print(f"Autocompletado:  '{resultado_1}'")

print("-" * 30)

frase_prueba_2 = "El perro"
resultado_2 = predecir_siguiente_palabra(modelo, tokenizer, frase_prueba_2, cantidad_palabras=4, max_sequence_len=max_sequence_len)
print(f"Usuario escribe: '{frase_prueba_2}'")
print(f"Autocompletado:  '{resultado_2}'")

Usuario escribe: 'El gato'
Autocompletado:  'El gato come pescado fresco'
------------------------------
Usuario escribe: 'El perro'
Autocompletado:  'El perro duerme todo el día'
